In [1]:
from __future__ import annotations

import operator
import os
import re
from datetime import date
from pathlib import Path


from typing import TypedDict , List , Annotated , Literal , Optional

from pydantic import BaseModel , Field

from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_mistralai import ChatMistralAI

from langchain_core.messages import SystemMessage , HumanMessage
from langchain_community.tools.tavily_search import TavilySearchResults 

from dotenv import load_dotenv
from langgraph.graph import StateGraph , START , END
from langgraph.types import Send

C:\Users\muham\AppData\Local\Temp\ipykernel_12876\3566628837.py:19: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [2]:
load_dotenv()

GROQ_KEY_1 = os.getenv("GROQ_API_KEY_1")
GROQ_KEY_2 = os.getenv("GROQ_API_KEY_2")

In [3]:
import os
from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY_1")
)

models = client.models.list()

for model in models.data:
    print(model.id)

meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-prompt-guard-2-22m
whisper-large-v3
openai/gpt-oss-120b
allam-2-7b
canopylabs/orpheus-v1-english
canopylabs/orpheus-arabic-saudi
openai/gpt-oss-safeguard-20b
whisper-large-v3-turbo
qwen/qwen3.8-27b
openai/gpt-oss-20b


In [4]:
class Task(BaseModel):
    id : int
    title : str
    
    goal : str = Field(
        ...,
        description =  "One sentence describing what the reader should be able to do/understand after this section."
    )
    
    bullets :List[str] = Field(
        ... ,
        min_length = 4 , 
        max_length = 6,
        description = "4-6 concreate , non-overlapping subpoints to cover in this section")
    
    target_words : int = Field(
        ...,
        description = "Target word count for this section (300-500)."
    )
    
    tags : List[str] = Field(default_factory = list)
    requires_research : bool = False
    requires_citations : bool = False
    requires_code : bool = False
    
    

In [5]:
class Plan(BaseModel):
    blog_title : str
    audience : str = Field(... , description ="Who is this blog for.")
    tone : str = Field(... , description = "Writing tone (e.g. , practical , crisp).")
    blog_kind : Literal['explainer' , 'tutorial' , 'news_roundup' , 'comparison' , 'system_design'] = 'explainer'
    constraints : List[str] = Field(default_factory = list)
    tasks : List[Task]

In [6]:
class EvidenceItem(BaseModel):
    title : str
    url : str
    published_at : Optional[str] = None
    snippet : Optional[str] = None
    source : Optional[str] = None

In [7]:
class RouterDecision(BaseModel):
    needs_research : bool
    mode : Literal['closed_book' , 'hybrid' , 'open_book']
    queries : List[str] = Field(default_factory = list)

In [8]:
class EvidencePack(BaseModel):
    evidence : List[EvidenceItem] = Field(default_factory = list)

In [9]:
class ImageSpec(BaseModel):
    placeholder: str = Field(
        description="Exact placeholder: [[IMAGE_1]], [[IMAGE_2]], or [[IMAGE_3]]"
    )

    section_title: str = Field(
        description="Exact Markdown section heading where the image should be inserted"
    )

    filename: str = Field(
        description="Safe image filename ending with .png or .jpg"
    )

    prompt: str = Field(
        description="Detailed prompt for generating the technical image"
    )

    caption: str = Field(
        description="Short caption explaining the image"
    )

    alt: str = Field(
        description="Short accessibility description of the image"
    )

In [10]:
class GlobalImagePlan(BaseModel):
    images: List[ImageSpec] = Field(
        default_factory=list,
        description="Maximum 3 technically useful images"
    )

In [11]:
class State(TypedDict):
    topic : str
    
    mode : str
    needs_research : bool
    queries : List[str]
    evidence : List[EvidenceItem]
    plan : Optional[Plan]
    
    as_of : str
    recency_days:int
    
    # The content which each worker node will give 
    sections: Annotated[List[tuple[int, str]], operator.add]
    
    #reducer/image
    merged_md :str
    md_with_placeholders : str
    image_specs : List[dict]
    
    
    final : str

In [12]:
planner_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_KEY_1
)

worker_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    api_key=GROQ_KEY_2
)

worker_gemini = ChatGoogleGenerativeAI(
    model = "gemini-3.1-flash-lite",
    temperature = 0.2
    
)

In [13]:
ROUTER_SYSTEM = """You are a routing module for a technical blog planner.

Decide whether web research is needed BEFORE planning.

Modes:
- closed_book (needs_research=false):
  Evergreen topics where correctness does not depend on recent facts (concepts, fundamentals).
- hybrid (needs_research=true):
  Mostly evergreen but needs up-to-date examples/tools/models to be useful.
- open_book (needs_research=true):
  Mostly volatile: weekly roundups, "this week", "latest", rankings, pricing, policy/regulation.

If needs_research=true:
- Output 2-3 high-signal queries.
- Queries should be scoped and specific (avoid generic queries like just "AI" or "LLM").
- If user asked for "last week/this week/latest", reflect that constraint IN THE QUERIES.
"""

def router_node(state : State)->dict:
    
    topic = state['topic']
    decider = planner_llm.with_structured_output(RouterDecision)
    decision = decider.invoke(
        [
            SystemMessage(content = ROUTER_SYSTEM),
            HumanMessage(content = f"Topic : {topic}")
        ]
    )
    
    return {
        "needs_research" : decision.needs_research , 
        "mode" : decision.mode,
        "queries" : decision.queries,
    }
    
def route_next(state : State)->str:
    return "research" if state['needs_research'] else 'orchestrator'
    

In [14]:
# -----------------------------
# 4) Research (Tavily) 
# -----------------------------
def _tavily_search(query: str, max_results: int = 5) -> List[dict]:

    tool = TavilySearchResults(max_results=max_results)
    results = tool.invoke({"query": query})

    normalized = []

    # Make sure the outer result is a list
    if not isinstance(results, list):
        print("Unexpected Tavily result type:", type(results))
        return normalized

    for r in results:

        # Ignore anything that isn't a dictionary
        if not isinstance(r, dict):
            print("Skipping unexpected result:", repr(r))
            continue

        normalized.append({
            "title": r.get("title", ""),
            "url": r.get("url", ""),
            "snippet": r.get("content", "") or r.get("snippet", ""),
            "published_at": (
                r.get("published_date")
                or r.get("published_at")
            ),
            "source": r.get("source"),
        })

    return normalized


RESEARCH_SYSTEM = """You are a research synthesizer for technical writing.

Given raw web search results, produce a deduplicated list of EvidenceItem objects.

Rules:
- Only include items with a non-empty url.
- Prefer relevant + authoritative sources (company blogs, docs, reputable outlets).
- If a published date is explicitly present in the result payload, keep it as YYYY-MM-DD.
  If missing or unclear, set published_at=null. Do NOT guess.
- Keep snippets short.
- Deduplicate by URL.
"""

def research_node(state: State) -> dict:

    # Keep research small
    queries = (state.get("queries", []) or [])[:2]

    raw_results: List[dict] = []

    for q in queries:
        raw_results.extend(
            _tavily_search(q, max_results=3)
        )

    if not raw_results:
        return {"evidence": []}

    # Deduplicate by URL
    dedup = {}

    for r in raw_results:
        url = r.get("url")

        if not url:
            continue

        dedup[url] = EvidenceItem(
            title=r.get("title", ""),
            url=url,
            snippet=(r.get("snippet", "") or "")[:1000],
            published_at=r.get("published_at"),
            source=r.get("source")
        )

    return {
        "evidence": list(dedup.values())[:6]
    }

In [15]:
ORCH_SYSTEM = """You are a senior technical writer and developer advocate.
Your job is to produce a highly actionable outline for a technical blog post.

Hard requirements:
- Create 3-5 sections (tasks) suitable for the topic and audience.
- Each task must include:
  1) goal (1 sentence)
  2) 4-6 bullets that are concrete, specific, and non-overlapping
  3) target word count (300-500)

Quality bar:
- Assume the reader is a developer; use correct terminology.
- Bullets must be actionable: build/compare/measure/verify/debug.
- Ensure the overall plan includes at least 2 of these somewhere:
  * minimal code sketch / MWE (set requires_code=True for that section)
  * edge cases / failure modes
  * performance/cost considerations
  * security/privacy considerations (if relevant)
  * debugging/observability tips

Grounding rules:
- Mode closed_book: keep it evergreen; do not depend on evidence.
- Mode hybrid:
  - Use evidence for up-to-date examples (models/tools/releases) in bullets.
  - Mark sections using fresh info as requires_research=True and requires_citations=True.
- Mode open_book:
  - Set blog_kind = "news_roundup".
  - Every section is about summarizing events + implications.
  - DO NOT include tutorial/how-to sections unless user explicitly asked for that.
  - If evidence is empty or insufficient, create a plan that transparently says "insufficient sources"
    and includes only what can be supported.

IMPORTANT JSON SAFETY RULES:
- The output is used as JSON structured data.
- Do NOT use LaTeX notation in any string.
- Do NOT use backslashes in task titles, goals, or bullets.
- Do NOT write expressions such as \\(, \\), \\text, \\sqrt, \\frac, or \\big.
- For mathematical formulas, use plain text instead.
- Example: write "Attention(Q,K,V) = softmax(QK^T / sqrt(d_k))V"
  instead of using LaTeX.
- Use normal ASCII characters for mathematical notation whenever possible.

Output must strictly match the Plan schema.
"""


def orchestrator_node(state : State)->dict:
    planner = planner_llm.with_structured_output(Plan)
    
    evidence = state.get("evidence", [])
    mode = state.get("mode", "closed_book")
    
    plan = planner.invoke(
        [
            SystemMessage(content = ORCH_SYSTEM),
            HumanMessage(content = (
                    f"Topic: {state['topic']}\n"
                    f"Mode: {mode}\n\n"
                    f"Evidence (ONLY use for fresh claims; may be empty):\n"
                    f"{[e.model_dump() for e in evidence][:8]}"))
        ]
    )
    return {'plan':plan}

In [16]:
def fanout(state: State):
    return [
        Send(
            "worker",
            {
                "task": task.model_dump(),
                "topic": state["topic"],
                "mode": state["mode"],
                "plan": state["plan"].model_dump(),
                "evidence": [
                    e.model_dump()
                    for e in state.get("evidence", [])
                ],
            },
        )
        for task in state["plan"].tasks
    ]
    
    

In [17]:
WORKER_SYSTEM = """You are a senior technical writer and developer advocate.
Write ONE section of a technical blog post in Markdown.

Hard constraints:
- Follow the provided Goal and cover ALL Bullets in order (do not skip or merge bullets).
- Stay close to Target words (±15%).
- Output ONLY the section content in Markdown (no blog title H1, no extra commentary).
- Start with a '## <Section Title>' heading.

Scope guard:
- If blog_kind == "news_roundup": do NOT turn this into a tutorial/how-to guide.
  Do NOT teach web scraping, RSS, automation, or "how to fetch news" unless bullets explicitly ask for it.
  Focus on summarizing events and implications.

Grounding policy:
- If mode == open_book:
  - Do NOT introduce any specific event/company/model/funding/policy claim unless it is supported by provided Evidence URLs.
  - For each event claim, attach a source as a Markdown link: ([Source](URL)).
  - Only use URLs provided in Evidence. If not supported, write: "Not found in provided sources."
- If requires_citations == true:
  - For outside-world claims, cite Evidence URLs the same way.
- Evergreen reasoning is OK without citations unless requires_citations is true.

Code:
- If requires_code == true, include at least one minimal, correct code snippet relevant to the bullets.

Style:
- Short paragraphs, bullets where helpful, code fences for code.
- Avoid fluff/marketing. Be precise and implementation-oriented.
"""


def worker_node(payload: dict) -> dict:
    
    task = Task(**payload["task"])
    plan = Plan(**payload["plan"])
    evidence = [EvidenceItem(**e) for e in payload.get("evidence", [])]
    topic = payload["topic"]
    mode = payload.get("mode", "closed_book")

    bullets_text = "\n- " + "\n- ".join(task.bullets)

    evidence_text = ""
    if evidence:
        evidence_text = "\n".join(
            f"- {e.title} | {e.url} | {e.published_at or 'date:unknown'}".strip()
            for e in evidence[:3]
        )

    section_md = worker_llm.invoke(
        [
            SystemMessage(content=WORKER_SYSTEM),
            HumanMessage(
                content=(
                    f"Blog title: {plan.blog_title}\n"
                    f"Audience: {plan.audience}\n"
                    f"Tone: {plan.tone}\n"
                    f"Blog kind: {plan.blog_kind}\n"
                    f"Constraints: {plan.constraints}\n"
                    f"Topic: {topic}\n"
                    f"Mode: {mode}\n\n"
                    f"Section title: {task.title}\n"
                    f"Goal: {task.goal}\n"
                    f"Target words: {task.target_words}\n"
                    f"Tags: {task.tags}\n"
                    f"requires_research: {task.requires_research}\n"
                    f"requires_citations: {task.requires_citations}\n"
                    f"requires_code: {task.requires_code}\n"
                    f"Bullets:{bullets_text}\n\n"
                    f"Evidence (ONLY use these URLs when citing):\n{evidence_text}\n"
                )
            ),
        ]
    ).text.strip()

    return {"sections": [(task.id, section_md)]}
      
    

In [18]:
def merge_content(state: State) -> dict:

    plan = state["plan"]
    assert plan is not None

    ordered_sections = [
        md
        for _, md in sorted(
            state["sections"],
            key=lambda x: x[0]
        )
    ]

    body = "\n\n".join(ordered_sections).strip()

    merged_md = (
        f"# {plan.blog_title}\n\n"
        f"{body}\n"
    )

    return {
        "merged_md": merged_md
    }

In [19]:
DECIDE_IMAGES_SYSTEM = """
You are an expert technical editor.

Your task is to decide whether technical images or diagrams
would materially improve understanding of the blog.

Rules:

1. Maximum 3 images.
2. Do not create decorative images.
3. Prefer:
   - architecture diagrams
   - workflow diagrams
   - system diagrams
   - comparison matrices
   - technical charts
4. Every image must provide meaningful technical value.
5. Every image must be based ONLY on information present
   in the provided blog.
6. Never invent:
   - benchmark scores
   - prices
   - model capabilities
   - parameter counts
   - dates
   - performance numbers
7. If the blog does not contain reliable numerical data,
   do NOT propose a quantitative chart.
8. section_title MUST exactly match an existing Markdown
   heading from the blog.
9. Use placeholders exactly:
   [[IMAGE_1]]
   [[IMAGE_2]]
   [[IMAGE_3]]
10. Each placeholder can be used only once.
11. Return ONLY GlobalImagePlan.
12. Do NOT return the blog.
13. Do NOT reproduce the Markdown.
14. Do NOT add explanations outside the structured output.

If no useful image is needed, return:

{
    "images": []
}
"""

def decide_images(state: State) -> dict:

    planner = planner_llm.with_structured_output(
        GlobalImagePlan
    )

    merged_md = state["merged_md"]
    plan = state["plan"]

    assert plan is not None

    image_plan = planner.invoke(
        [
            SystemMessage(
                content=DECIDE_IMAGES_SYSTEM
            ),

            HumanMessage(
                content=(
                    f"Blog kind: {plan.blog_kind}\n"
                    f"Topic: {state['topic']}\n\n"
                    f"Blog content:\n\n"
                    f"{merged_md}"
                )
            ),
        ]
    )

    # Hard limit in Python as well.
    images = image_plan.images[:3]

    return {
        "image_specs": [
            image.model_dump()
            for image in images
        ]
    }
    

In [20]:
def insert_image_placeholders(state: State) -> dict:

    md = state["merged_md"]

    image_specs = (
        state.get("image_specs", [])
        or []
    )

    for spec in image_specs:

        section_title = spec["section_title"]
        placeholder = spec["placeholder"]

        # Check that the section exists.
        if section_title not in md:

            print(
                f"WARNING: Section not found: "
                f"{section_title}"
            )

            continue

        # Find the section.
        section_start = md.find(
            section_title
        )

        # Find the next ## heading.
        next_section = md.find(
            "\n## ",
            section_start + len(section_title)
        )

        if next_section == -1:

            # This is the final section.
            insert_position = len(md)

        else:

            insert_position = next_section

        md = (
            md[:insert_position]
            + f"\n\n{placeholder}\n"
            + md[insert_position:]
        )

    return {
        "md_with_placeholders": md
    }

In [21]:
def safe_image_filename(filename: str) -> str:

    filename = Path(filename).name

    filename = re.sub(
        r"[^a-zA-Z0-9._-]",
        "_",
        filename
    )

    return filename

In [22]:
def safe_filename(name: str) -> str:

    name = re.sub(
        r'[<>:"/\\|?*]',
        '',
        name
    )

    name = re.sub(
        r'\s+',
        '_',
        name.strip()
    )

    return name 

In [23]:
def _gemini_generate_image_bytes(prompt: str) -> bytes:

    from google import genai
    from google.genai import types

    api_key = os.environ.get(
        "GOOGLE_API_KEY"
    )

    if not api_key:
        raise RuntimeError(
            "GOOGLE_API_KEY is not set."
        )

    client = genai.Client(
        api_key=api_key
    )

    response = client.models.generate_content(
        model="gemini-2.5-flash-image",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_modalities=["IMAGE"]
        ),
    )

    parts = getattr(
        response,
        "parts",
        None
    )

    if not parts and getattr(
        response,
        "candidates",
        None
    ):
        try:
            parts = (
                response
                .candidates[0]
                .content
                .parts
            )
        except Exception:
            parts = None

    if not parts:
        raise RuntimeError(
            "No image content returned."
        )

    for part in parts:

        inline = getattr(
            part,
            "inline_data",
            None
        )

        if inline and getattr(
            inline,
            "data",
            None
        ):
            return inline.data

    raise RuntimeError(
        "No inline image bytes found."
    )

In [24]:
def generate_and_place_images(
    state: State
) -> dict:

    plan = state["plan"]
    assert plan is not None

    md = (
        state.get("md_with_placeholders")
        or state["merged_md"]
    )

    image_specs = (
        state.get("image_specs", [])
        or []
    )

    # No images
    if not image_specs:

        filename = (
            safe_filename(plan.blog_title)
            + ".md"
        )

        Path(filename).write_text(
            md,
            encoding="utf-8"
        )

        return {
            "final": md
        }

    images_dir = Path("images")
    images_dir.mkdir(
        exist_ok=True
    )

    for spec in image_specs:

        placeholder = spec["placeholder"]

        filename = safe_image_filename(
            spec["filename"]
        )

        out_path = (
            images_dir / filename
        )

        # Generate only if image doesn't already exist.
        if not out_path.exists():

            try:

                img_bytes = (
                    _gemini_generate_image_bytes(
                        spec["prompt"]
                    )
                )

                out_path.write_bytes(
                    img_bytes
                )

            except Exception as e:

                print(
                    f"Image generation failed "
                    f"for {filename}: {e}"
                )

                fallback = (
                    f"> **[IMAGE GENERATION FAILED]** "
                    f"{spec.get('caption', '')}\n>\n"
                    f"> **Alt:** "
                    f"{spec.get('alt', '')}\n"
                )

                md = md.replace(
                    placeholder,
                    fallback
                )

                continue

        img_md = (
            f"![{spec['alt']}]"
            f"(images/{filename})\n"
            f"*{spec['caption']}*"
        )

        md = md.replace(
            placeholder,
            img_md
        )

    filename = (
        safe_filename(plan.blog_title)
        + ".md"
    )

    Path(filename).write_text(
        md,
        encoding="utf-8"
    )

    print(
        "Final file:",
        Path(filename).resolve()
    )

    return {
        "final": md
    }
  

   

In [25]:
def run(topic: str):
    out = app.invoke(
        {
            "topic": topic,
            "mode": "",
            "needs_research": False,
            "queries": [],
            "evidence": [],
            "plan": None,
            "as_of": str(date.today()),
            "recency_days": 30,
            "sections": [],
            "merged_md": "",
            "md_with_placeholders": "",
            "image_specs": [],
            "final": "",
        }
    )

    return out

In [26]:
g = StateGraph(State)

g.add_node(
    "router",
    router_node
)

g.add_node(
    "research",
    research_node
)

g.add_node(
    "orchestrator",
    orchestrator_node
)

g.add_node(
    "worker",
    worker_node
)

g.add_node(
    "merge_content",
    merge_content
)

g.add_node(
    "decide_images",
    decide_images
)

g.add_node(
    "insert_image_placeholders",
    insert_image_placeholders
)

g.add_node(
    "generate_and_place_images",
    generate_and_place_images
)


g.add_edge(
    START,
    "router"
)

g.add_conditional_edges(
    "router",
    route_next,
    {
        "research": "research",
        "orchestrator": "orchestrator"
    }
)

g.add_edge(
    "research",
    "orchestrator"
)

g.add_conditional_edges(
    "orchestrator",
    fanout,
    ["worker"]
)

g.add_edge(
    "worker",
    "merge_content"
)

g.add_edge(
    "merge_content",
    "decide_images"
)

g.add_edge(
    "decide_images",
    "insert_image_placeholders"
)

g.add_edge(
    "insert_image_placeholders",
    "generate_and_place_images"
)

g.add_edge(
    "generate_and_place_images",
    END
)

app = g.compile()

In [27]:
run("State of Multimodel LLMs in 2026")

C:\Users\muham\AppData\Local\Temp\ipykernel_12876\2961142850.py:6: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool = TavilySearchResults(max_results=max_results)
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Image generation failed for model_comparison.png: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-flash-preview-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-flash-preview-image\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.5-flash-preview-image\nPlease retry in 40.353810832s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'u

{'topic': 'State of Multimodel LLMs in 2026',
 'mode': 'open_book',
 'needs_research': True,
 'queries': ['What multimodal large language models were released or updated in 2026, including architecture details and capabilities?',
  '2026 benchmark results and performance comparisons for multimodal LLMs on tasks like visual question answering, image captioning, and video understanding',
  'Current pricing, API access, and major cloud provider offerings for multimodal LLM services in 2026'],
 'evidence': [EvidenceItem(title='30 of the best large language models in 2026', url='https://www.techtarget.com/whatis/feature/12-of-the-best-large-language-models', published_at=None, snippet="### Falcon\n\nFalcon transformer-based models developed by the Technology Innovation Institute is open source and has multilingual capabilities. Falcon 2 is available in an 11-billion-parameter version that provides multimodal capabilities for both text and vision. Falcon 3 is available in several sizes rangi